# Capítulo 10 · Evaluar una aplicación con LLM

Un banco de 50 preguntas con respuestas de referencia, tres versiones del asistente de los capítulos 5 y 6, y cuatro formas de medir: comprobación automática de claves, similitud con la referencia, un modelo como juez y etiquetas manuales. Además, el intervalo de confianza de una diferencia y el análisis de errores.

**Necesitas la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*. Es el cuaderno más pesado del libro: unas 800 llamadas a los modelos de lenguaje. En una GPU de escritorio tarda unos trece minutos, y bastante más en Colab. Sin GPU no es práctico.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

📘 El libro, en Amazon: [tapa blanda](https://www.amazon.es/dp/B0HMXCBCTN) · [Kindle](https://www.amazon.es/dp/B0HMXG3NXM)

## 1. Instalar, cargar los modelos de búsqueda y los datos

In [ ]:
!pip install -q rank_bm25

In [ ]:
"""Capítulo 10: evaluar una aplicación con LLM. En GPU (en CPU, muy lento)."""
import json
import random
import re
import unicodedata
import urllib.request
from pathlib import Path

import numpy as np
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO = "Qwen/Qwen2.5-1.5B-Instruct"       # el del asistente
JUECES = ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen3-1.7B"]
MODELO_EMBEDDINGS = "intfloat/multilingual-e5-base"
MODELO_REORDENACION = "BAAI/bge-reranker-v2-m3"
DATOS = Path("datos")
REPOSITORIO = "https://raw.githubusercontent.com/HernanDiaz/llm-engineering/main/"
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"



def cargar_lenguaje(nombre):
    """Un modelo de lenguaje y su tokenizador, en la GPU si la hay."""
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(nombre, dtype=torch.float32)
    return tok, modelo.to(DISPOSITIVO).eval()


def descargar(archivo):
    """Un archivo de la carpeta datos; en Colab, se descarga del repositorio."""
    ruta = DATOS / archivo
    if not ruta.exists():
        DATOS.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORIO + ruta.as_posix(), ruta)
    texto = ruta.read_text(encoding="utf-8")
    return json.loads(texto) if archivo.endswith(".json") else texto

In [ ]:
embeddings = SentenceTransformer(MODELO_EMBEDDINGS, device=DISPOSITIVO)
reordenador = CrossEncoder(MODELO_REORDENACION, device=DISPOSITIVO)
banco = descargar("banco_evaluacion.json")
print(len(banco), "preguntas. Una de ellas:")
print(json.dumps(banco[13], ensure_ascii=False, indent=1))

## 2. El asistente de los capítulos 5 y 6

In [ ]:
def base_de_conocimiento():
    """Preguntas frecuentes, condiciones generales y catálogo."""
    fragmentos = [{"fuente": f"Preguntas frecuentes, {p['id']}",
                   "texto": p["pregunta"] + " " + p["respuesta"]}
                  for p in descargar("preguntas_frecuentes.json")["preguntas"]]
    for seccion in re.split(r"\n(?=## )", descargar("condiciones_generales.md"))[1:]:
        titulo, texto = seccion.strip().split("\n", 1)
        fragmentos.append({"fuente": "Condiciones generales, " + titulo[3:],
                           "texto": texto.strip()})
    for p in descargar("catalogo.json"):
        fragmentos.append({"fuente": f"Catálogo, {p['ref']}", "texto": (
            f"{p['nombre']} (referencia {p['ref']}), {p['categoria']}, "
            f"{p['precio']} euros. {p['descripcion']}")})
    return fragmentos


def palabras(texto):
    """Palabras para BM25: minúsculas, sin tildes, referencias enteras."""
    texto = "".join(c for c in unicodedata.normalize("NFD", texto.lower())
                    if unicodedata.category(c) != "Mn")
    return re.findall(r"[a-z]{2}-\d{4}|[a-z0-9]{3,}", texto)


class Buscador:
    """Embeddings, BM25 y reordenación, como en el capítulo 6."""

    def __init__(self, fragmentos):
        self.fragmentos = fragmentos
        self.textos = [f["texto"] for f in fragmentos]
        self.matriz = embeddings.encode(["passage: " + t for t in self.textos],
                                        normalize_embeddings=True)
        self.bm25 = BM25Okapi([palabras(t) for t in self.textos])

    def semantico(self, consulta):
        v = embeddings.encode(["query: " + consulta], normalize_embeddings=True)[0]
        return list(np.argsort(-(self.matriz @ v)))

    def mejorado(self, consulta, n=20):
        puntos = self.bm25.get_scores(palabras(consulta))
        bm25 = [i for i in np.argsort(-puntos, kind="stable") if puntos[i] > 0]
        rrf = {}
        for lista in (self.semantico(consulta), bm25):
            for pos, i in enumerate(lista, 1):
                rrf[i] = rrf.get(i, 0) + 1 / (60 + pos)
        candidatos = sorted(rrf, key=rrf.get, reverse=True)[:n]
        notas = reordenador.predict([(consulta, self.textos[i]) for i in candidatos])
        return [candidatos[i] for i in np.argsort(-notas)]

In [ ]:
NO_LO_SE = "No tengo esa información; paso tu consulta al equipo."


SISTEMA_RAG = (
    "Eres el asistente de atención al cliente de Ciclos Norte, una tienda de "
    "bicicletas de Oviedo. Responde a la pregunta del cliente en español, en dos "
    "o tres frases, usando solo la información de los documentos que se te dan. "
    "Después de cada dato, indica entre corchetes el número del documento del que "
    "sale, por ejemplo [2]. Si los documentos no contienen la respuesta, responde "
    f"exactamente: «{NO_LO_SE}» No inventes nada.")


SISTEMA_COMPROBAR = (
    "Decide si los documentos tienen alguna relación con la pregunta del cliente. "
    "Responde «no» solo si ningún documento trata del tema de la pregunta; en caso "
    "contrario, responde «sí».")


def instruccion_rag(pregunta, documentos):
    """Los documentos numerados y la pregunta, como en el capítulo 5."""
    lista = "\n\n".join(f"[{i}] ({d['fuente']}) {d['texto']}"
                        for i, d in enumerate(documentos, 1))
    return f"Documentos:\n\n{lista}\n\nPregunta del cliente: {pregunta}"


EJEMPLOS_RAG = [
    (instruccion_rag("¿Tenéis cascos para mi hijo de cinco años?", [
        {"fuente": "Catálogo, cascos", "texto": "Cascos infantiles en las tallas "
         "XS (46-50 cm) y S (50-54 cm), recomendados de 3 a 8 años."},
        {"fuente": "Condiciones generales, 3. Precios",
         "texto": "Todos los precios incluyen el IVA."}]),
     "Sí, tenemos cascos infantiles en las tallas XS y S, recomendados de 3 a 8 "
     "años [1]."),
    (instruccion_rag("¿Vendéis patinetes eléctricos?", [
        {"fuente": "Catálogo, bicicletas eléctricas", "texto": "Bicicletas "
         "eléctricas urbanas y de montaña, con autonomía de 60 a 120 km."},
        {"fuente": "Preguntas frecuentes, 99", "texto": "¿Alquiláis bicicletas? "
         "Sí, bicicletas de paseo y eléctricas por días."}]),
     NO_LO_SE),
]


def generar(tok, modelo, sistema, usuario, ejemplos=(), max_tokens=200):
    """Respuesta de un modelo, con ejemplos opcionales y temperatura 0."""
    mensajes = [{"role": "system", "content": sistema}]
    for p, r in ejemplos:
        mensajes += [{"role": "user", "content": p},
                     {"role": "assistant", "content": r}]
    mensajes.append({"role": "user", "content": usuario})
    texto = tok.apply_chat_template(mensajes, tokenize=False,
                                    add_generation_prompt=True,
                                    enable_thinking=False)
    entrada = tok(texto, return_tensors="pt").to(DISPOSITIVO)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=max_tokens,
                                 do_sample=False, temperature=None, top_p=None,
                                 top_k=None, pad_token_id=tok.eos_token_id)
    respuesta = tok.decode(salida[0, entrada["input_ids"].shape[1]:],
                           skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", respuesta, flags=re.S).strip()


def asistente(pregunta, recuperar, comprobar=False, k=3):
    """El asistente con una forma de recuperar y, si se pide, la comprobación
    previa del capítulo 5 (en su versión menos estricta)."""
    documentos = [buscador.fragmentos[i] for i in recuperar(pregunta)[:k]]
    instruccion = instruccion_rag(pregunta, documentos)
    if comprobar and generar(tok, modelo, SISTEMA_COMPROBAR, instruccion,
                             max_tokens=3).lower().startswith("no"):
        return NO_LO_SE, documentos
    return generar(tok, modelo, SISTEMA_RAG, instruccion, EJEMPLOS_RAG), documentos

## 3. Cuatro formas de medir

In [ ]:
def contiene(texto, claves):
    """¿Contiene el texto alguna de las claves?"""
    return any(c.lower() in texto.lower() for c in claves)


def comprobacion_automatica(caso, respuesta):
    """Correcta si contiene un dato clave sin abstenerse, o si se abstiene
    cuando la pregunta no tiene respuesta."""
    se_abstiene = NO_LO_SE[:25].lower() in respuesta.lower()
    if caso["claves"] is None:
        return se_abstiene
    return contiene(respuesta, caso["claves"]) and not se_abstiene


def similitud(respuesta, referencia):
    """Similitud del coseno entre los embeddings de la respuesta y la referencia."""
    v = embeddings.encode(["query: " + respuesta, "query: " + referencia],
                          normalize_embeddings=True)
    return float(v[0] @ v[1])


JUEZ_CORRECCION = (
    "Eres un evaluador estricto. Compara la respuesta de un asistente con la "
    "respuesta de referencia, que es la correcta. La respuesta es correcta si "
    "contiene la información esencial de la referencia y no la contradice ni "
    "añade datos falsos; el estilo y la longitud no importan. Si la referencia "
    "dice que no hay información, la respuesta solo es correcta si tampoco da "
    "ninguna. Responde solo «correcta» o «incorrecta».")


JUEZ_FIDELIDAD = (
    "Eres un evaluador estricto. Decide si todo lo que afirma la respuesta está "
    "respaldado por los documentos. Si la respuesta dice que no tiene la "
    "información, cuenta como respaldada. Responde solo «sí» o «no».")


def juzgar(tok_j, juez, caso, respuesta, documentos):
    """Las dos notas del juez: corrección frente a la referencia y fidelidad a
    los documentos."""
    texto = (f"Pregunta: {caso['pregunta']}\n\nReferencia: {caso['referencia']}"
             f"\n\nRespuesta del asistente: {respuesta}")
    correcta = generar(tok_j, juez, JUEZ_CORRECCION, texto, max_tokens=4)
    docs = "\n".join(f"- {d['texto']}" for d in documentos)
    fiel = generar(tok_j, juez, JUEZ_FIDELIDAD,
                   f"Documentos:\n{docs}\n\nRespuesta: {respuesta}", max_tokens=3)
    return {"correcta": correcta.lower().lstrip("«").startswith("correcta"),
            "fiel": fiel.lower().lstrip("«").startswith("s")}

## 4. ¿Es real la diferencia?

In [ ]:
def diferencia_pareada(a, b, repeticiones=10000, semilla=0):
    """Diferencia de aciertos entre dos sistemas en las mismas preguntas, con
    un intervalo de confianza del 95 % por remuestreo (bootstrap)."""
    azar = random.Random(semilla)
    n = len(a)
    difs = []
    for _ in range(repeticiones):
        muestra = [azar.randrange(n) for _ in range(n)]
        difs.append(sum(b[i] - a[i] for i in muestra))
    difs.sort()
    return {"diferencia": sum(b) - sum(a),
            "intervalo": (difs[int(0.025 * repeticiones)],
                          difs[int(0.975 * repeticiones)]),
            "solo_a": sum(x and not y for x, y in zip(a, b)),
            "solo_b": sum(y and not x for x, y in zip(a, b))}


def acuerdo(automaticas, manuales):
    """Proporción de casos en que una medida coincide con las etiquetas manuales."""
    return sum(x == y for x, y in zip(automaticas, manuales)) / len(manuales)

In [ ]:
# Un ejemplo con datos inventados: dos sistemas que aciertan 30 y 33 de 50.
import random
azar = random.Random(1)
a = [azar.random() < 0.6 for _ in range(50)]
b = [x or azar.random() < 0.15 for x in a]
print(sum(a), sum(b), diferencia_pareada(a, b))

## 5. Evaluar los tres sistemas

Es la parte lenta.

In [ ]:
SISTEMAS = {
    "A. RAG del capítulo 5": dict(recuperar="semantico", comprobar=False),
    "B. Recuperación del capítulo 6": dict(recuperar="mejorado", comprobar=False),
    "C. B + comprobación previa": dict(recuperar="mejorado", comprobar=True),
}


def ejecutar():
    """Responde al banco con los tres sistemas, los juzga y los compara."""
    global buscador, tok, modelo
    banco = descargar("banco_evaluacion.json")
    buscador = Buscador(base_de_conocimiento())
    tok, modelo = cargar_lenguaje(MODELO)
    resultados = {}
    for nombre, opciones in SISTEMAS.items():
        recuperar = getattr(buscador, opciones["recuperar"])
        filas = []
        for caso in banco:
            respuesta, docs = asistente(caso["pregunta"], recuperar,
                                        opciones["comprobar"])
            filas.append({
                "pregunta": caso["pregunta"], "tipo": caso["tipo"],
                "respuesta": respuesta, "documentos": docs,
                "automatica": comprobacion_automatica(caso, respuesta),
                "recuperada": caso["claves"] is None or any(
                    contiene(d["texto"], caso["claves"]) for d in docs),
                "similitud": similitud(respuesta, caso["referencia"])})
        resultados[nombre] = filas
        print(f"{nombre}: {sum(f['automatica'] for f in filas)}/50 correctas")
    del modelo
    for nombre_juez in JUECES:
        tok_j, juez = cargar_lenguaje(nombre_juez)
        for filas in resultados.values():
            for caso, f in zip(banco, filas):
                f[nombre_juez] = juzgar(tok_j, juez, caso, f["respuesta"],
                                        f["documentos"])
        del juez
        if DISPOSITIVO == "cuda":
            torch.cuda.empty_cache()
    for f in (f for filas in resultados.values() for f in filas):
        f["documentos"] = [d["fuente"] for d in f["documentos"]]
    a, b = (resultados[n] for n in list(SISTEMAS)[:2])
    resultados["A frente a B"] = diferencia_pareada(
        [f["automatica"] for f in a], [f["automatica"] for f in b])
    print(resultados["A frente a B"])
    return resultados

In [ ]:
resultados = ejecutar()

## 6. Comparar con las etiquetas manuales

Las etiquetas del libro corresponden a las respuestas de su ejecución; si las tuyas cambian, revísalas y corrígelas.

In [ ]:
ruta = Path("latex/tema_10/etiquetas_manuales.json")   # en el repositorio
if not ruta.exists():                                   # en Colab, se descarga
    ruta = Path("etiquetas_manuales.json")
    urllib.request.urlretrieve(REPOSITORIO + "latex/tema_10/etiquetas_manuales.json", ruta)
manual = json.loads(ruta.read_text(encoding="utf-8"))["correctas"]
B = resultados["B. Recuperación del capítulo 6"]
medidas = {"Comprobación automática": [f["automatica"] for f in B],
           "Similitud >= 0,85": [f["similitud"] >= 0.85 for f in B]}
for juez in JUECES:
    medidas[f"Juez {juez}"] = [f[juez]["correcta"] for f in B]
for nombre, v in medidas.items():
    print(f"{nombre:38s} {sum(v):2d} correctas, coincide en {acuerdo(v, manual):.0%}")

Los errores del sistema B, según las etiquetas manuales, con si se recuperó el fragmento correcto:

In [ ]:
for caso, f, bien in zip(banco, B, manual):
    if not bien:
        print(f"[{caso['tipo']}] {caso['pregunta']}")
        print("    recuperado:", f["recuperada"], "| respuesta:", f["respuesta"][:150])

## Ejercicios

1. Explica con tus palabras la diferencia entre corrección y fidelidad. Pon un ejemplo de respuesta correcta pero no fiel y otro de respuesta fiel pero incorrecta.

2. Etiqueta a mano las 50 respuestas del sistema A con el mismo criterio del capítulo. ¿Cuántas son correctas? ¿Coincide tu comparación de A y B con la de la comprobación automática?

3. Revisa las etiquetas manuales del capítulo. ¿Estás de acuerdo con todas? Cambia las que no te convenzan y recalcula el acuerdo de cada medida.

4. Mejora la instrucción del juez Qwen3-1.7B para que sea más estricto, por ejemplo con dos ejemplos resueltos de respuestas incorrectas. ¿Mejora su acuerdo con las etiquetas?

5. Activa el razonamiento del juez Qwen3-1.7B (`enable_thinking=True`, con más tokens de salida). ¿Juzga mejor? ¿Cuánto más tarda?

6. Calcula el intervalo de confianza de la diferencia entre B y C. ¿Puedes afirmar que la comprobación previa mejora el sistema?

7. Duplica el banco de pruebas copiando las 50 preguntas, y calcula de nuevo el intervalo entre A y B. ¿Por qué es engañoso? ¿Qué habría que hacer de verdad para tener un banco más grande?

8. Usa el juez de fidelidad para encontrar las respuestas del sistema B que afirman algo que no está en los documentos. ¿Coinciden con las invenciones que encontraste a mano?

9. Añade diez preguntas nuevas al banco, de los tipos en los que más falla el sistema, con su referencia y sus claves. ¿Cambia el análisis de errores?

10. **Informe de evaluación.** Escribe un informe de una página sobre el asistente: banco de pruebas, medidas usadas y su fiabilidad, resultados de los tres sistemas, análisis de errores y recomendaciones para la siguiente versión.

## Referencias

Modelos [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B), [multilingual-e5-base](https://huggingface.co/intfloat/multilingual-e5-base) y [bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3). Modelo como juez: [Zheng y colaboradores (2023)](https://arxiv.org/abs/2306.05685). [RAGAS](https://arxiv.org/abs/2309.15217) (Es y colaboradores, 2023). Lecciones 11.10 y 05.27 y proyectos 19.70 a 19.75 de [AI Engineering from Scratch](https://github.com/rohitg00/ai-engineering-from-scratch). La tienda Ciclos Norte y sus documentos son ficticios. Código y experimentos propios.